# Training and Evaluation in one Notebook for One Model-Database Pair

# To check before running
1. Check class names for your event log in the **p2pencoder.py** ( *{event_log_name}encoder.py* )
2. Check the Axioms in **axiombuilder.py**
3. make sure you have done the declare mining on the event log and have a valid **ltn_rows_path**

In [1]:
event_log_name = "huge"
if event_log_name is None:
    raise ValueError("Please set the event_log_name variable to the name of the event log you want to use.")
ltn_rows_path = f"{event_log_name}_ltn_rows.pkl"
print(f"Event log name {event_log_name}")
print(f"LTN Rows path {ltn_rows_path}")
# starting time


Event log name huge
LTN Rows path huge_ltn_rows.pkl


In [2]:
# import tensorflow as tf
# physical_devices = tf.config.list_physical_devices('GPU')
# print(physical_devices)
# if len(physical_devices) > 0:
#     tf.config.experimental.set_memory_growth(physical_devices[0], True)
#     print("GPU found")
#     print("Memory growth set")
# else:
#     print("No GPU found")

In [3]:
import arrow
import socket
from sqlalchemy.orm import Session
from tqdm.notebook import tqdm

from april.anomalydetection import *
from april.database import EventLog
from april.database import Model
from april.database import get_engine
from april.dataset import Dataset
from april.fs import DATE_FORMAT
from april.fs import get_event_log_files

import itertools

from sklearn import metrics


from april.anomalydetection import BINet
from april.anomalydetection.utils import label_collapse
from april.database import Evaluation
from april.hugeevaluator import Evaluator
from april.fs import get_model_files
from april.fs import PLOT_DIR

import matplotlib.pyplot as plt
import numpy as np
np.random.seed(0)

import pandas as pd
import seaborn as sns
from sqlalchemy.orm import Session
import scikit_posthocs as sp

from april.database import get_engine
from april.fs import PLOT_DIR
from april.utils import microsoft_colors, prettify_dataframe, cd_plot, get_cd
from april.enums import Base, Strategy, Heuristic

sns.set_style('white')
pd.set_option('display.max_rows', 50)
%config InlineBackend.figure_format = 'retina'
print(huge_leaky_row_classes)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU found
Memory growth set
Creating Evaluation table
[<class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-10'>, <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-25'>, <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-50'>, <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-100'>, <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-150'>, <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-200'>, <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-250'>, <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-300'>]


In [52]:
dataset = f"{event_log_name}-0.3-1"
out_dir = PLOT_DIR / f'{event_log_name}_evaluations_both_{arrow.now().format("YYYY-MM-DD-HH-mm-ss")}'
eval_file = out_dir / f'{event_log_name}_fraction_evaluations.pkl'
csv_file = out_dir / f'{event_log_name}_fraction_evaluations.csv'
excel_file = out_dir / f'{event_log_name}_fraction_evaluations.xlsx'
model_folder = r"D:\LTNcoder\.out\models"
db = r"D:\LTNcoder\.out\april.db"

# create out_dir if it does not exist
if not out_dir.exists():
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {out_dir}")
from april.utils import delete_all_files_in_folder, delete_evaluation_and_model_tables
delete_all_files_in_folder(model_folder)
delete_evaluation_and_model_tables(db)
start_time = arrow.now("Europe/Berlin")


Created directory: d:\LTNcoder\.out\plots\huge_evaluations_both_2025-08-09-18-14-57
Deleted all rows from Evaluation and Model tables.


# Training

In [53]:
def fit_and_save(dataset_name, ad, ad_kwargs=None, fit_kwargs=None):
    if ad_kwargs is None:
        ad_kwargs = {}
    if fit_kwargs is None:
        fit_kwargs = {}

    # Save start time
    start_time = arrow.now()

    # Dataset
    dataset = Dataset(dataset_name)

    # AD
    ad = ad(**ad_kwargs)

    # Train and save
    ad.fit(dataset, **fit_kwargs)
    file_name = f'{dataset_name}_{ad.abbreviation}_{start_time.format(DATE_FORMAT)}'
    model_file = ad.save(file_name)

    # Save end time
    end_time = arrow.now()

    # Cache result
    Evaluator(model_file.str_path).cache_result()

    # Calculate training time in seconds
    training_time = (end_time - start_time).total_seconds()

    # Write to database
    engine = get_engine()
    session = Session(engine)

    session.add(Model(creation_date=end_time.datetime,
                      algorithm=ad.name,
                      training_duration=training_time,
                      file_name=model_file.file,
                      training_event_log_id=EventLog.get_id_by_name(dataset_name),
                      training_host=socket.gethostname(),
                      hyperparameters=str(dict(**ad_kwargs, **fit_kwargs))))
    session.commit()
    session.close()

    if isinstance(ad, NNAnomalyDetector):
        from keras.backend import clear_session
        clear_session()
    pass

In [54]:
ads = [
        dict(ad=HugeDAE, fit_kwargs=dict(epochs=6, batch_size=100)),
    ] + \
    [
        dict(ad=LEAKY_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100)) for LEAKY_ROW_CLASS 
        in huge_leaky_row_classes
    ] + \
    [
        dict(ad=LTN_ROW_CLASS, fit_kwargs=dict(epochs=6, batch_size=100, epochs_ltn=3))
        for LTN_ROW_CLASS in huge_ltn_row_classes
    ]
print(ads)
for ad in tqdm(ads, desc="Fitting ADs"):
    fit_and_save(dataset, **ad)


[{'ad': <class 'april.anomalydetection.hugeencoder.HugeDAE'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-10'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-25'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-50'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-100'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-150'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-200'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-250'>, 'fit_kwargs': {'epochs': 6, 'batch_size': 100}}, {'ad': <class 'april.anomalydetection.hugeencoder.Hu

Fitting ADs:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 1/6
43/43 [==============================] - 1s 16ms/step - loss: 0.1796 - accuracy: 0.1417 - val_loss: 0.0150 - val_accuracy: 0.0000e+00
Epoch 2/6
43/43 [==============================] - 0s 7ms/step - loss: 0.0055 - accuracy: 0.2943 - val_loss: 0.0041 - val_accuracy: 0.0000e+00
Epoch 3/6
43/43 [==============================] - 0s 7ms/step - loss: 0.0041 - accuracy: 0.3661 - val_loss: 0.0040 - val_accuracy: 0.0000e+00
Epoch 4/6
43/43 [==============================] - 0s 7ms/step - loss: 0.0041 - accuracy: 0.4182 - val_loss: 0.0039 - val_accuracy: 0.0043
Epoch 5/6
43/43 [==============================] - 0s 7ms/step - loss: 0.0040 - accuracy: 0.4622 - val_loss: 0.0039 - val_accuracy: 0.0150
Epoch 6/6
43/43 [==============================] - 0s 7ms/step - loss: 0.0039 - accuracy: 0.4643 - val_loss: 0.0037 - val_accuracy: 0.0855
d:\LTNcoder\.out\models\huge-0.3-1_hugedae_20250809-181457.415575.keras
Loading model huge-0.3-1_hugedae_20250809-181457.415575 / <april.fs.ModelFile obj

In [55]:
print(AD) #Evaluator dependso on AD

{'binetv0': <class 'april.anomalydetection.binet.binet.BINetv0'>, 'binetv1': <class 'april.anomalydetection.binet.binet.BINetv1'>, 'binetv2': <class 'april.anomalydetection.binet.binet.BINetv2'>, 'binetv3': <class 'april.anomalydetection.binet.binet.BINetv3'>, 'likelihood': <class 'april.anomalydetection.boehmer.BoehmerLikelihoodAnomalyDetector'>, 'dae': <class 'april.anomalydetection.autoencoder.DAE'>, 'daeltn': <class 'april.anomalydetection.autoencoder.DAELTN'>, 'daeltnfrozen': <class 'april.anomalydetection.autoencoder.DAELTNFROZEN'>, 'hugedae': <class 'april.anomalydetection.hugeencoder.HugeDAE'>, 'hugedae-leaky-10': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-10'>, 'hugedae-leaky-100': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-100'>, 'hugedae-leaky-150': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-150'>, 'hugedae-leaky-200': <class 'april.anomalydetection.hugeencoder.HugeDAE-Leaky-200'>, 'hugedae-leaky-25': <class 'april.anomalydetection.h

# Evaluation

In [56]:
heuristics = [h for h in Heuristic.keys() if h not in [Heuristic.DEFAULT, Heuristic.MANUAL, Heuristic.RATIO,
                                                       Heuristic.MEDIAN, Heuristic.MEAN]]
params = [(Base.SCORES, Heuristic.DEFAULT, Strategy.SINGLE), *itertools.product([Base.SCORES], heuristics, Strategy.keys())]

In [57]:
def _evaluate(params):
    # print(f"Evaluating {params}...")
    e, base, heuristic, strategy = params

    session = Session(get_engine())
    model = session.query(Model).filter_by(file_name=e.model_file.name).first()
    session.close()
    # print(f"Model {model} loaded from Session.")

    if Model is None:
        print(f"Models from session: {model}")
    # Generate evaluation frames
    y_pred = e.binarizer.binarize(base=base, heuristic=heuristic, strategy=strategy, go_backwards=False)
    y_true = e.binarizer.get_targets()

    evaluations = []
    for axis in [0, 1, 2]:
        # print(f"Evaluating axis {axis}...")
        for i, attribute_name in enumerate(e.dataset.attribute_keys):
            # print(f"Evaluating attribute {attribute_name}...")
            def get_evaluation(label, precision, recall, f1):
                return Evaluation(model_id=model.id, file_name=model.file_name,
                                  label=label, perspective=perspective, attribute_name=attribute_name,
                                  axis=axis, base=base, heuristic=heuristic, strategy=strategy,
                                  precision=precision, recall=recall, f1=f1)

            perspective = 'Control Flow' if i == 0 else 'Data'
            if i > 0 and not e.ad_.supports_attributes:
                # print(f"Skipping attribute {attribute_name} for model {model} as it does not support attributes.")
                evaluations.append(get_evaluation('Normal', 0.0, 0.0, 0.0))
                evaluations.append(get_evaluation('Anomaly', 0.0, 0.0, 0.0))
            else:
                # print(f"Evaluating attribute {attribute_name} for model {model}...")
                yp = label_collapse(y_pred[:, :, i:i + 1], axis=axis).compressed()
                yt = label_collapse(y_true[:, :, i:i + 1], axis=axis).compressed()
                p, r, f, _ = metrics.precision_recall_fscore_support(yt, yp, labels=[0, 1])
                evaluations.append(get_evaluation('Normal', p[0], r[0], f[0]))
                evaluations.append(get_evaluation('Anomaly', p[1], r[1], f[1]))

    return evaluations

def evaluate(model_name):
    print(f"Evaluating {model_name}...")
    e = Evaluator(model_name)
    print(f"{e} loaded.")
    # print attributes of e
    print(f"e.model_file: {e.model_file}")
    print(f"e.model_name: {e.model_name}")
    print(f"e.eventlog_name: {e.eventlog_name}")
    print(f"e.dataset: {e.dataset}")
    print(f"e.result: {e.result}")
    
    
    _params = []
    for base, heuristic, strategy in params:
        if e.dataset.num_attributes == 1 and strategy in [Strategy.ATTRIBUTE, Strategy.POSITION_ATTRIBUTE]:
            continue
        if isinstance(e.ad_, BINet) and e.ad_.version == 0:
            continue
        if heuristic is not None and heuristic not in e.ad_.supported_heuristics:
            continue
        if strategy is not None and strategy not in e.ad_.supported_strategies:
            continue
        if base is not None and base not in e.ad_.supported_bases:
            continue
        # print(f"Adding parameters: {e}, {base}, {heuristic}, {strategy}")
        _params.append([e, base, heuristic, strategy])
    
    print(f"{_params} parameters to evaluate.")

    return [_e for p in _params for _e in _evaluate(p)]

In [58]:
models = sorted([m.name for m in get_model_files() if m.p == 0.3])# and 'real' in m.name])
print(f"Available Models: {models}")
evaluations = []
for model in tqdm(models, desc='Evaluate'):
    e = evaluate(model)
    evaluations.append(e)
# Write to database
session = Session(get_engine())
for e in evaluations:
    session.bulk_save_objects(e)
    session.commit()
session.close()

Available Models: ['huge-0.3-1_hugedae-leaky-100_20250809-181510.846344', 'huge-0.3-1_hugedae-leaky-10_20250809-181501.102960', 'huge-0.3-1_hugedae-leaky-150_20250809-181514.066806', 'huge-0.3-1_hugedae-leaky-200_20250809-181517.282836', 'huge-0.3-1_hugedae-leaky-250_20250809-181520.852921', 'huge-0.3-1_hugedae-leaky-25_20250809-181504.321855', 'huge-0.3-1_hugedae-leaky-300_20250809-181524.165484', 'huge-0.3-1_hugedae-leaky-50_20250809-181507.539576', 'huge-0.3-1_hugedae_20250809-181457.415575', 'huge-0.3-1_hugeltnfrozen-100_20250809-181626.376509', 'huge-0.3-1_hugeltnfrozen-10_20250809-181527.421108', 'huge-0.3-1_hugeltnfrozen-150_20250809-181646.440243', 'huge-0.3-1_hugeltnfrozen-200_20250809-181718.205387', 'huge-0.3-1_hugeltnfrozen-250_20250809-181749.686543', 'huge-0.3-1_hugeltnfrozen-25_20250809-181546.844095', 'huge-0.3-1_hugeltnfrozen-300_20250809-181818.814277', 'huge-0.3-1_hugeltnfrozen-50_20250809-181606.618564']


Evaluate:   0%|          | 0/17 [00:00<?, ?it/s]

Evaluating huge-0.3-1_hugedae-leaky-100_20250809-181510.846344...
Loading model huge-0.3-1_hugedae-leaky-100_20250809-181510.846344 / <april.fs.ModelFile object at 0x0000029AB6AE06D0> for event log huge-0.3-1 at path d:\LTNcoder\.out\models\huge-0.3-1_hugedae-leaky-100_20250809-181510.846344.keras
Self.ad_: <april.anomalydetection.hugeencoder.HugeDAE-Leaky-100 object at 0x0000029AB6AE0C70>
<april.hugeevaluator.Evaluator object at 0x0000029AB6AE0340> loaded.
e.model_file: d:\LTNcoder\.out\models\huge-0.3-1_hugedae-leaky-100_20250809-181510.846344.keras
e.model_name: huge-0.3-1_hugedae-leaky-100_20250809-181510.846344
e.eventlog_name: huge-0.3-1
Filtering dataset to 326 LTN rows.
Indices: [1, 7, 16, 35, 36, 54, 59, 71, 73, 75, 77, 78, 87, 95, 141, 155, 163, 165, 193, 202, 205, 210, 211, 254, 273, 302, 309, 325, 360, 385, 415, 422, 426, 434, 443, 491, 496, 500, 503, 528, 555, 562, 572, 581, 584, 624, 626, 640, 641, 654, 668, 698, 720, 746, 788, 803, 823, 826, 899, 902, 904, 911, 931, 937,

In [59]:

session = Session(get_engine())
evaluations = session.query(Evaluation).all()
rows = []

for ev in tqdm(evaluations):
    # print(f"Evaluation: {ev}")
    m = ev.model
    # print(f"Model: {m}")
    el = ev.model.training_event_log
    # print(f"Event log: {el}")
    rows.append([m.file_name, m.creation_date, m.hyperparameters, m.training_duration, m.training_host, m.algorithm, 
                 el.name, el.base_name, el.percent_anomalies, el.number,
                 ev.axis, ev.base, ev.heuristic, ev.strategy, ev.label, ev.attribute_name, ev.perspective, ev.precision, ev.recall, ev.f1])
session.close()
columns = ['file_name', 'date', 'hyperparameters', 'training_duration', 'training_host', 'ad',
           'dataset_name', 'process_model', 'noise', 'dataset_id',
           'axis', 'base', 'heuristic', 'strategy', 'label', 'attribute_name', 'perspective', 'precision', 'recall', 'f1']
evaluation = pd.DataFrame(rows, columns=columns)

evaluation.to_pickle(eval_file)

  0%|          | 0/4896 [00:00<?, ?it/s]

In [60]:
synth_datasets = ['paper', 'p2p', 'small', 'medium', 'large', 'huge', 'gigantic', 'wide']
bpic_datasets = ['bpic12', 'bpic13', 'bpic15', 'bpic17']
anonymous_datasets = ['real']
datasets = synth_datasets + bpic_datasets + anonymous_datasets
dataset_types = ['Synthetic', 'Real-life']

orig_ads = [ad['ad'].__name__ for ad in ads if "DAE" in ad['ad'].__name__]
new_ads = [ad['ad'].__name__ for ad in ads if "DAE" not in ad['ad'].__name__]
ads = orig_ads + new_ads

heuristics = [r'$best$', r'$default$', r'$elbow_\downarrow$', r'$elbow_\uparrow$', 
              r'$lp_\leftarrow$', r'$lp_\leftrightarrow$', r'$lp_\rightarrow$']
print(ads)

['HugeDAE', 'HugeDAE-Leaky-10', 'HugeDAE-Leaky-25', 'HugeDAE-Leaky-50', 'HugeDAE-Leaky-100', 'HugeDAE-Leaky-150', 'HugeDAE-Leaky-200', 'HugeDAE-Leaky-250', 'HugeDAE-Leaky-300', 'HugeLTNFROZEN-10', 'HugeLTNFROZEN-25', 'HugeLTNFROZEN-50', 'HugeLTNFROZEN-100', 'HugeLTNFROZEN-150', 'HugeLTNFROZEN-200', 'HugeLTNFROZEN-250', 'HugeLTNFROZEN-300']


In [61]:
evaluation = evaluation.query(f'ad in {ads} and label == "Anomaly"')

In [62]:
evaluation['perspective-label'] = evaluation['perspective'] + '-' + evaluation['label']
evaluation['attribute_name-label'] = evaluation['attribute_name'] + '-' + evaluation['label']
evaluation['dataset_type'] = 'Synthetic'
evaluation.loc[evaluation['process_model'].str.contains('bpic'), 'dataset_type'] = 'Real-life'
evaluation.loc[evaluation['process_model'].str.contains('real'), 'dataset_type'] = 'Real-life'

In [63]:
_filtered_evaluation = evaluation.query(f'ad in {ads} and (strategy == "{Strategy.ATTRIBUTE}"'
                                       f' or (strategy == "{Strategy.SINGLE}" and process_model == "bpic12")'
                                       f' or (strategy == "{Strategy.SINGLE}" and ad == "Naive+"))')

In [64]:
filtered_evaluation = _filtered_evaluation.query(f'heuristic == "{Heuristic.DEFAULT}"'
                                                 f' or (heuristic == "{Heuristic.LP_MEAN}" and ad in {orig_ads})'
                                                 f' or (heuristic == "{Heuristic.LP_LEFT}" and ad in {new_ads})'
                                                )

In [65]:
df = filtered_evaluation.query('axis == 0')
df = prettify_dataframe(df)
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name', 'perspective'])[['precision', 'recall', 'f1']].mean().reset_index()
df = df.groupby(['axis', 'process_model', 'dataset_name', 'ad', 'file_name'])[['precision', 'recall', 'f1']].mean().reset_index()
df['f1'] = 2 * df['recall'] * df['precision'] / (df['recall'] + df['precision'])

df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model', 'dataset_name'], values=['precision', 'recall', 'f1'])
df = df.fillna(0)
df = df.stack(1).stack(1).reset_index()
df.to_excel(str(out_dir / 'table.xlsx'), index=False)

# drop rows in column "axis" which have value "Attribute"
df = df.query('axis != "Attribute"')

# df = pd.pivot_table(df, index=['axis', 'ad'], columns=['process_model'], values=['precision', 'recall', 'f1'], aggfunc=np.mean)

df.to_excel(str(excel_file), index=False)
df.to_csv(str(csv_file), index=False)
print(df)

    axis                 ad process_model dataset_name        f1  precision  \
0   Case            HugeDAE          Huge   huge-0.3-1  0.524852   0.701613   
1   Case   HugeDAE-Leaky-10          Huge   huge-0.3-1  0.373426   0.800000   
2   Case  HugeDAE-Leaky-100          Huge   huge-0.3-1  0.323850   0.800000   
3   Case  HugeDAE-Leaky-150          Huge   huge-0.3-1  0.384189   0.760000   
4   Case  HugeDAE-Leaky-200          Huge   huge-0.3-1  0.305359   0.785714   
5   Case   HugeDAE-Leaky-25          Huge   huge-0.3-1  0.412608   0.868421   
6   Case  HugeDAE-Leaky-250          Huge   huge-0.3-1  0.456114   0.854167   
7   Case  HugeDAE-Leaky-300          Huge   huge-0.3-1  0.356274   0.789474   
8   Case   HugeDAE-Leaky-50          Huge   huge-0.3-1  0.547362   0.707692   
9   Case   HugeLTNFROZEN-10          Huge   huge-0.3-1  0.380596   0.823529   
10  Case  HugeLTNFROZEN-100          Huge   huge-0.3-1  0.594173   0.655914   
11  Case  HugeLTNFROZEN-150          Huge   huge-0.3

C:\Users\devas\AppData\Local\Temp\ipykernel_35220\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()
C:\Users\devas\AppData\Local\Temp\ipykernel_35220\661170537.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(1).stack(1).reset_index()


In [66]:
display(df)

,axis,ad,process_model,dataset_name,f1,precision,recall
0,Case,HugeDAE,Huge,huge-0.3-1,0.524852,0.701613,0.419233
1,Case,HugeDAE-Leaky-10,Huge,huge-0.3-1,0.373426,0.800000,0.243558
2,Case,HugeDAE-Leaky-100,Huge,huge-0.3-1,0.323850,0.800000,0.203017
3,Case,HugeDAE-Leaky-150,Huge,huge-0.3-1,0.384189,0.760000,0.257071
4,Case,HugeDAE-Leaky-200,Huge,huge-0.3-1,0.305359,0.785714,0.189503
5,Case,HugeDAE-Leaky-25,Huge,huge-0.3-1,0.412608,0.868421,0.270585
6,Case,HugeDAE-Leaky-250,Huge,huge-0.3-1,0.456114,0.854167,0.311125
7,Case,HugeDAE-Leaky-300,Huge,huge-0.3-1,0.356274,0.789474,0.230044
8,Case,HugeDAE-Leaky-50,Huge,huge-0.3-1,0.547362,0.707692,0.446260
9,Case,HugeLTNFROZEN-10,Huge,huge-0.3-1,0.380596,0.823529,0.247486


# End

In [67]:
end_time = arrow.now("Europe/Berlin")
print(f"Start time: {start_time}")
print(f"End time: {end_time}")
print(f"Duration: {end_time - start_time}")

Start time: 2025-08-09T18:14:57.359889+02:00
End time: 2025-08-09T18:19:33.818932+02:00
Duration: 0:04:36.459043
